# Multimodal RAG with Image Captioning — From First Principles

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/rag/multi_model_rag_with_captioning.ipynb)

Traditional RAG only indexes text. Real documents are multimodal: charts, diagrams, screenshots, tables, and figures often hold critical information. One simple way to bring those assets into a text-centric RAG pipeline is **captioning**: convert the visual content into descriptive text, embed that text, and retrieve it like any other passage.

This notebook builds that pipeline from scratch. The central idea is not that captioning is perfect. It is that captioning is a practical bridge between image-heavy documents and a text-based retrieval stack.

## The Multimodal Challenge

If a report includes a chart showing that evening peaks occur after solar output falls, a text-only retriever may miss that evidence completely. OCR can recover visible text, but it cannot reliably express visual relationships like bars, trends, arrows, and layouts.

Captioning compresses that visual structure into a textual description. Some information is lost, but much more becomes retrievable than with a pure text pipeline.

## Captioning as an Image-to-Text Bridge

```
image
  -> captioning model
  -> caption text
  -> text embedding
  -> unified retrieval with ordinary text chunks
```

The elegance of this approach is operational simplicity. We do not need a special multimodal vector database to understand the pipeline. We only need a caption generator and a unified text index.

## How Vision-Language Models Work

Captioning models are a specific type of **vision-language model** (VLM). Understanding their architecture clarifies both their power and their limitations for RAG pipelines.

### Encoder-Decoder Architecture

Most modern captioning models follow an encoder-decoder pattern:

1. **Vision Encoder** — A pretrained image model (e.g., ViT) processes the raw pixels into a sequence of visual tokens (patch embeddings).
2. **Bridge Module** — An intermediate component aligns the visual representation with the language model's embedding space.
3. **Language Decoder** — A text model (often a transformer decoder) generates the caption token by token, attending to the bridged visual features.

```
┌─────────────┐     ┌──────────────────┐     ┌──────────────────┐
│  Raw Image   │────>│  Vision Encoder  │────>│  Patch Embeddings│
│  (pixels)    │     │  (ViT / CNN)     │     │  (N × d_v)       │
└─────────────┘     └──────────────────┘     └────────┬─────────┘
                                                       │
                                                       ▼
                                              ┌──────────────────┐
                                              │  Bridge / Adapter│
                                              │  (Q-Former, Lin. │
                                              │   Proj, etc.)    │
                                              └────────┬─────────┘
                                                       │
                                                       ▼
                                              ┌──────────────────┐
                                              │ Language Decoder  │
                                              │ (GPT / OPT)      │
                                              │ Cross-Attention   │
                                              │ over visual tokens│
                                              └──────────────────┘
                                                       │
                                                       ▼
                                                 "a bar chart
                                                  showing solar,
                                                  wind, and battery"
```

### Cross-Attention: Where Vision Meets Language

The key mechanism is **cross-attention**. At each decoding step, the language model attends not only to previously generated tokens (self-attention) but also to the visual embeddings. This lets the decoder *look at* relevant image regions while choosing the next word.

### BLIP-2 and the Q-Former

BLIP-2 (the family behind our `blip-image-captioning-base` model) introduced the **Q-Former** (Querying Transformer) as its bridge module:

- A set of **learned query vectors** attend to the frozen image encoder's output via cross-attention.
- These queries extract a fixed number of visual tokens (typically 32) regardless of image resolution.
- The resulting tokens are projected into the language model's embedding space.

This design has two practical consequences for RAG:

| Property | Implication for RAG |
|----------|-------------------|
| Fixed-length output | Captions are compact — easy to embed and index |
| Learned queries | The model selects *what to describe* — it can miss details not captured by the query vectors |
| Frozen encoder + decoder | Fast inference, but limited adaptation to domain-specific visuals |

Understanding this architecture helps explain why captions are sometimes generic or miss fine-grained details — the Q-Former bottleneck compresses the image into a small number of tokens before the language model ever sees it.

## Environment Setup

All notebooks in this overhaul use the same minimal native stack:

- **Qwen/Qwen3-8B** for generation
- **BAAI/bge-base-en-v1.5** for embeddings
- **FAISS** for similarity search
- **Google Drive** for persistent Colab caching

The goal is to keep the pipeline transparent. Every major component is visible as plain Python functions instead of framework abstractions.

In [ ]:
!pip install -q transformers>=4.51.0 accelerate bitsandbytes torch sentence-transformers faiss-cpu numpy Pillow
print("Installed core native RAG dependencies.")

In [ ]:
import time
import math
import json
import re
from collections import defaultdict, Counter

import numpy as np
import torch
from google.colab import drive
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

drive.mount("/content/drive")
CACHE_DIR = "/content/drive/MyDrive/models"
MODEL_NAME = "Qwen/Qwen3-8B"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    quantization_config=quantization_config,
    cache_dir=CACHE_DIR,
    torch_dtype="auto"
)
print("Loaded", MODEL_NAME)

In [ ]:
import faiss
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer("BAAI/bge-base-en-v1.5", cache_folder=CACHE_DIR)
print("Loaded embedding model:", embed_model.__class__.__name__)

In [ ]:
def generate(prompt, max_new_tokens=220, temperature=0.2):
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        top_k=20,
        temperature=temperature,
        do_sample=temperature > 0,
    )
    answer_ids = outputs[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(answer_ids, skip_special_tokens=True).strip()

print("Generation helper ready.")

In [ ]:
from PIL import Image, ImageDraw, ImageFont
from transformers import AutoProcessor, AutoModelForVision2Seq

CAPTION_MODEL = "Salesforce/blip-image-captioning-base"
caption_processor = AutoProcessor.from_pretrained(CAPTION_MODEL, cache_dir=CACHE_DIR)
caption_model = AutoModelForVision2Seq.from_pretrained(CAPTION_MODEL, cache_dir=CACHE_DIR, device_map="auto")
print("Loaded captioning model:", CAPTION_MODEL)

## Generating Synthetic Images

To keep the notebook self-contained, we create simple images with PIL. These stand in for the kinds of visual artifacts often found in real enterprise documents: trend charts, labeled diagrams, and status dashboards.

In [ ]:
def make_chart_image():
    image = Image.new("RGB", (480, 320), "white")
    draw = ImageDraw.Draw(image)
    draw.rectangle((50, 40, 430, 260), outline="black", width=2)
    bars = [("Solar", 140, "#f4b400"), ("Wind", 110, "#4285f4"), ("Battery", 70, "#34a853")]
    x = 90
    for label, height, color in bars:
        draw.rectangle((x, 260 - height, x + 60, 260), fill=color, outline="black")
        draw.text((x, 270), label, fill="black")
        x += 110
    draw.text((120, 10), "Evening Flexibility Resources", fill="black")
    return image

def make_diagram_image():
    image = Image.new("RGB", (480, 320), "white")
    draw = ImageDraw.Draw(image)
    draw.ellipse((40, 110, 120, 190), outline="black", width=3)
    draw.rectangle((190, 80, 290, 220), outline="black", width=3)
    draw.polygon([(370, 150), (320, 110), (320, 190)], outline="black", fill="#d9ead3")
    draw.text((55, 145), "Solar", fill="black")
    draw.text((205, 145), "Battery", fill="black")
    draw.text((335, 145), "Grid", fill="black")
    draw.line((120, 150, 190, 150), fill="black", width=3)
    draw.line((290, 150, 320, 150), fill="black", width=3)
    draw.text((140, 125), "charges", fill="black")
    draw.text((295, 125), "discharges", fill="black")
    return image

images = {
    "chart": make_chart_image(),
    "diagram": make_diagram_image(),
}

for name, image in images.items():
    print(name, image.size)
    display(image)

## Captioning the Images

The caption model turns each image into a text description. That text becomes just another retrievable item in the corpus.

In [ ]:
def caption_image(image):
    inputs = caption_processor(images=image, return_tensors="pt").to(caption_model.device)
    output_ids = caption_model.generate(**inputs, max_new_tokens=40)
    return caption_processor.batch_decode(output_ids, skip_special_tokens=True)[0].strip()

captions = {name: caption_image(image) for name, image in images.items()}
for name, caption in captions.items():
    print(name, "->", caption)

## Inspecting Caption Quality

Before we trust captions as retrieval evidence, we should inspect them critically. Caption models can exhibit several failure modes:

- **Hallucinated details** — describing objects or colors not present in the image
- **Missed objects** — omitting prominent visual elements
- **Generic descriptions** — producing vague captions that lose discriminating detail
- **Misread relationships** — incorrectly describing spatial or causal links

Below we display each image alongside its caption and annotate potential quality issues.

In [ ]:
print("=" * 70)
print("CAPTION QUALITY INSPECTION")
print("=" * 70)

# Ground-truth descriptions we know from construction
ground_truth = {
    "chart": "A bar chart titled 'Evening Flexibility Resources' with three bars: Solar (tallest, gold), Wind (medium, blue), Battery (shortest, green).",
    "diagram": "A flow diagram: Solar (circle) -> charges -> Battery (rectangle) -> discharges -> Grid (triangle). Shows energy flow direction.",
}

for name, caption in captions.items():
    print(f"\n--- {name.upper()} ---")
    print(f"Generated caption : {caption}")
    print(f"Ground truth      : {ground_truth[name]}")
    display(images[name])

    # Simple quality checks
    gt_lower = ground_truth[name].lower()
    cap_lower = caption.lower()
    gt_keywords = {"chart": ["bar", "solar", "wind", "battery", "evening"],
                   "diagram": ["solar", "battery", "grid", "flow", "charge"]}
    found = [kw for kw in gt_keywords[name] if kw in cap_lower]
    missed = [kw for kw in gt_keywords[name] if kw not in cap_lower]
    print(f"Keywords found     : {found}")
    print(f"Keywords missed    : {missed}")
    coverage = len(found) / max(len(gt_keywords[name]), 1)
    print(f"Keyword coverage   : {coverage:.0%}")

print("\n" + "=" * 70)
print("TAKEAWAY: Caption quality directly limits retrieval quality.")
print("A caption that misses 'battery' means queries about batteries")
print("will never retrieve this image's evidence.")
print("=" * 70)

## Text Chunks and Caption Chunks in One Index

We now create a unified corpus containing both ordinary text passages and caption-derived passages. The retriever will not care where the text came from; it only cares that the text is semantically relevant.

In [ ]:
text_docs = [
    {"id": "t1", "kind": "text", "text": "Evening peaks occur after solar generation declines, so fast flexibility resources become especially valuable in late afternoon and early night."},
    {"id": "t2", "kind": "text", "text": "Batteries respond quickly and can absorb midday solar oversupply before discharging during the evening ramp."},
    {"id": "t3", "kind": "text", "text": "Wind output often complements solar because it can remain strong at night or during seasonal weather systems."},
    {"id": "t4", "kind": "text", "text": "Grid operators value resources that provide fast response, flexible ramping, and support during evening peak periods."},
]

caption_docs = [
    {"id": f"img_{name}", "kind": "caption", "text": caption}
    for name, caption in captions.items()
]

multimodal_docs = text_docs + caption_docs
for row in multimodal_docs:
    print(row["id"], "|", row["kind"], "|", row["text"][:100])

In [ ]:
multi_embeddings = embed_model.encode(
    [row["text"] for row in multimodal_docs],
    normalize_embeddings=True,
)
multi_embeddings = np.asarray(multi_embeddings, dtype="float32")
multi_index = faiss.IndexFlatIP(multi_embeddings.shape[1])
multi_index.add(multi_embeddings)
print("Unified index size:", multi_index.ntotal)

## Embedding Space Analysis

Are caption embeddings and text embeddings interleaved in the vector space, or do they cluster separately? This matters because:

- If they cluster **separately**, a text-phrased query might systematically prefer text chunks over captions (or vice versa), reducing the benefit of a unified index.
- If they are **interleaved**, the retriever can naturally surface whichever evidence is most semantically relevant regardless of origin.

We compute pairwise cosine similarities within and across the two groups to find out.

In [ ]:
# Split embeddings back into text vs caption groups
n_text = len(text_docs)
n_caption = len(caption_docs)
text_vecs = multi_embeddings[:n_text]
caption_vecs = multi_embeddings[n_text:]

# Compute within-group and cross-group cosine similarities
# (embeddings are already L2-normalized, so dot product = cosine similarity)
text_text_sims = text_vecs @ text_vecs.T
caption_caption_sims = caption_vecs @ caption_vecs.T
text_caption_sims = text_vecs @ caption_vecs.T

# Extract off-diagonal similarities (exclude self-similarity of 1.0)
def off_diagonal_mean(matrix):
    n = matrix.shape[0]
    if n < 2:
        return float('nan')
    mask = ~np.eye(n, dtype=bool)
    return float(matrix[mask].mean())

stats = {
    "Text ↔ Text (avg cos sim)": off_diagonal_mean(text_text_sims),
    "Caption ↔ Caption (avg cos sim)": off_diagonal_mean(caption_caption_sims),
    "Text ↔ Caption (avg cos sim)": float(text_caption_sims.mean()),
}

print("Embedding Space Analysis")
print("=" * 50)
for label, val in stats.items():
    print(f"  {label:40s}: {val:.4f}")

print("\nPairwise Text ↔ Caption similarities:")
for i, td in enumerate(text_docs):
    for j, cd in enumerate(caption_docs):
        sim = float(text_caption_sims[i, j])
        print(f"  {td['id']} <-> {cd['id']}: {sim:.4f}")

print("\nInterpretation:")
avg_within = (stats['Text ↔ Text (avg cos sim)'] + stats['Caption ↔ Caption (avg cos sim)']) / 2
avg_cross = stats['Text ↔ Caption (avg cos sim)']
if avg_cross > avg_within * 0.8:
    print("  Caption and text embeddings are reasonably interleaved.")
    print("  The unified index should retrieve across both types effectively.")
else:
    print("  Caption and text embeddings show some clustering.")
    print("  Queries may have a bias toward one evidence type.")
    print("  Consider tuning caption style to match the text register.")

## Retrieval over Mixed Evidence

The question below is intentionally visual. A text-only index can discuss evening peaks, but the captions may carry extra structure about the chart or the flow diagram.

In [ ]:
def multimodal_retrieve(question, k=4):
    vec = embed_model.encode([question], normalize_embeddings=True)
    vec = np.asarray(vec, dtype="float32")
    scores, indices = multi_index.search(vec, k)
    return [
        {
            "score": float(score),
            "id": multimodal_docs[idx]["id"],
            "kind": multimodal_docs[idx]["kind"],
            "text": multimodal_docs[idx]["text"],
        }
        for score, idx in zip(scores[0], indices[0])
    ]

hits = multimodal_retrieve("Which resources in the chart help with evening flexibility?")
for hit in hits:
    print(hit["kind"], hit["id"], round(hit["score"], 3), "|", hit["text"])

## A Text-Only Baseline

To judge whether captioning helps, we compare the unified index to a text-only baseline built from the four text passages alone.

In [ ]:
text_embeddings = embed_model.encode([row["text"] for row in text_docs], normalize_embeddings=True)
text_embeddings = np.asarray(text_embeddings, dtype="float32")
text_index = faiss.IndexFlatIP(text_embeddings.shape[1])
text_index.add(text_embeddings)

def text_only_retrieve(question, k=4):
    vec = embed_model.encode([question], normalize_embeddings=True)
    vec = np.asarray(vec, dtype="float32")
    scores, indices = text_index.search(vec, min(k, len(text_docs)))
    return [text_docs[idx]["id"] for idx in indices[0]]

question = "What does the diagram suggest about the role of batteries between solar and the grid?"
print("Text-only:", text_only_retrieve(question))
print("Multimodal:", [row["id"] for row in multimodal_retrieve(question)])

## End-to-End Multimodal RAG

The generation step is still ordinary grounded generation. The difference is only in how the context was assembled.

In [ ]:
def multimodal_answer(question):
    hits = multimodal_retrieve(question, k=4)
    context = "\n\n".join([f"[{row['id']} | {row['kind']}] {row['text']}" for row in hits])
    prompt = f"""
Use the multimodal evidence below. Captions describe images and diagrams.
If the evidence is incomplete, say so.

Evidence:
{context}

Question: {question}

Answer:
""".strip()
    return {"hits": hits, "answer": generate(prompt, max_new_tokens=170)}

response = multimodal_answer("How do the chart and diagram together explain evening grid support?")
print("Evidence IDs:", [row["id"] for row in response["hits"]])
print("\nAnswer:\n", response["answer"])

## Multiple Query Examples

Different query types stress different parts of the multimodal pipeline. We test four categories:

1. **Factual** — Directly answerable from text passages
2. **Visual** — Requires information from image captions
3. **Reasoning** — Needs synthesis across text and visual evidence
4. **Unanswerable** — Not covered by any evidence (tests refusal behavior)

In [ ]:
test_queries = [
    {
        "type": "Factual",
        "query": "Why are fast flexibility resources valuable in the evening?",
    },
    {
        "type": "Visual",
        "query": "What three resources are shown in the bar chart?",
    },
    {
        "type": "Reasoning",
        "query": "How does the flow from solar to battery to grid help during evening peaks?",
    },
    {
        "type": "Unanswerable",
        "query": "What is the exact megawatt capacity of each resource?",
    },
]

for tq in test_queries:
    print("\n" + "=" * 70)
    print(f"QUERY TYPE: {tq['type']}")
    print(f"QUESTION  : {tq['query']}")
    print("-" * 70)
    result = multimodal_answer(tq["query"])
    print("Retrieved evidence:")
    for hit in result["hits"]:
        print(f"  [{hit['id']:10s} | {hit['kind']:7s} | score={hit['score']:.3f}] {hit['text'][:80]}")
    print(f"\nAnswer:\n{result['answer']}")
    print("=" * 70)

## Why Captioning Helps — and Where It Fails

Captioning helps because it converts otherwise invisible image content into retrievable language. But captioning is lossy. It can flatten layout, miss exact numeric values, and fail on complex tables or dense dashboards.

That means captioning is a bridge, not the end state of multimodal retrieval. It is often the right first implementation because it is simple and operationally clear.

In [ ]:
comparison_questions = [
    "Which resources in the chart help with evening flexibility?",
    "What role does the battery play between solar and the grid?",
    "Which evidence mentions nighttime complementarity?",
]

for q in comparison_questions:
    print("\nQUESTION:", q)
    print("TEXT-ONLY IDs:", text_only_retrieve(q))
    print("MULTIMODAL IDs:", [row["id"] for row in multimodal_retrieve(q)])

## System Design Tradeoffs

The multimodal captioning pipeline adds:

- a captioning model,
- image processing,
- and caption storage.

In return, it allows a plain text retriever to reason over images indirectly. For many teams that is a great tradeoff because it preserves the rest of the stack: embeddings, FAISS, and text-grounded generation remain unchanged.

In [ ]:
tradeoffs = [
    ["Implementation complexity", "Moderate", "One extra model, same retriever"],
    ["Image fidelity", "Medium", "Good for diagrams, weaker for dense tables"],
    ["Operational simplicity", "High", "Still a text index"],
    ["Best use case", "Mixed reports", "Documents with a few important figures"],
]

for row in tradeoffs:
    print(" | ".join(row))

## Performance Metrics

To quantify whether captioning actually improves retrieval, we compute **Precision@k** and **Recall** for a set of evaluation queries with known relevant documents. We compare the multimodal (text + captions) index against the text-only baseline.

In [ ]:
# Evaluation queries with ground-truth relevant document IDs
eval_queries = [
    {
        "query": "Which resources in the chart help with evening flexibility?",
        "relevant": {"t1", "t4", "img_chart"},
    },
    {
        "query": "What role does the battery play between solar and the grid?",
        "relevant": {"t2", "img_diagram"},
    },
    {
        "query": "Which evidence mentions nighttime complementarity?",
        "relevant": {"t3"},
    },
    {
        "query": "How does solar energy get stored and released?",
        "relevant": {"t2", "img_diagram"},
    },
]

def precision_at_k(retrieved_ids, relevant_ids, k):
    """Fraction of top-k retrieved docs that are relevant."""
    top_k = retrieved_ids[:k]
    return len(set(top_k) & relevant_ids) / k if k > 0 else 0.0

def recall(retrieved_ids, relevant_ids):
    """Fraction of relevant docs that appear in retrieved results."""
    if not relevant_ids:
        return 0.0
    return len(set(retrieved_ids) & relevant_ids) / len(relevant_ids)

In [ ]:
# Compute metrics for both systems across all eval queries
results = []

for eq in eval_queries:
    # Multimodal retrieval
    mm_hits = multimodal_retrieve(eq["query"], k=4)
    mm_ids = [h["id"] for h in mm_hits]

    # Text-only retrieval
    to_ids = text_only_retrieve(eq["query"], k=4)

    results.append({
        "query": eq["query"][:50] + "...",
        "relevant": eq["relevant"],
        "mm_ids": mm_ids,
        "to_ids": to_ids,
        "mm_p2": precision_at_k(mm_ids, eq["relevant"], 2),
        "mm_p4": precision_at_k(mm_ids, eq["relevant"], 4),
        "mm_recall": recall(mm_ids, eq["relevant"]),
        "to_p2": precision_at_k(to_ids, eq["relevant"], 2),
        "to_p4": precision_at_k(to_ids, eq["relevant"], 4),
        "to_recall": recall(to_ids, eq["relevant"]),
    })

# Print comparison table
print(f"{'Query':<55s} | {'P@2':>6s} {'P@4':>6s} {'Rec':>6s} | {'P@2':>6s} {'P@4':>6s} {'Rec':>6s}")
print(f"{'':55s} | {'--- Multimodal ---':>20s} | {'--- Text-Only ---':>20s}")
print("-" * 100)
for r in results:
    print(f"{r['query']:<55s} | {r['mm_p2']:6.2f} {r['mm_p4']:6.2f} {r['mm_recall']:6.2f} | {r['to_p2']:6.2f} {r['to_p4']:6.2f} {r['to_recall']:6.2f}")

# Averages
avg = lambda key: sum(r[key] for r in results) / len(results)
print("-" * 100)
print(f"{'AVERAGE':<55s} | {avg('mm_p2'):6.2f} {avg('mm_p4'):6.2f} {avg('mm_recall'):6.2f} | {avg('to_p2'):6.2f} {avg('to_p4'):6.2f} {avg('to_recall'):6.2f}")

print("\nConclusion:")
mm_avg_recall = avg('mm_recall')
to_avg_recall = avg('to_recall')
if mm_avg_recall > to_avg_recall:
    delta = mm_avg_recall - to_avg_recall
    print(f"  Multimodal recall is {delta:.0%} higher than text-only.")
    print("  Captioning successfully surfaces image-derived evidence.")
else:
    print("  Text-only recall matches or exceeds multimodal — captions may not help for these queries.")

## Key Takeaways

Multimodal RAG does not have to start with an exotic architecture. A captioning pipeline already demonstrates the core systems idea: **convert visual evidence into retrievable context and integrate it with text retrieval**.

That makes captioning an excellent educational stepping stone. It reveals both the power of multimodal augmentation and the precise places where more advanced visual retrieval methods become necessary.